# Golay Encoder

In [2]:
%load_ext autoreload
%autoreload 1
%aimport classes.GaloisField
%aimport classes.GolayDecoder

import numpy as np

from classes.GaloisField import *
from classes.GaloisPoly  import *
from classes.GolayEncoder import GolayEncoder
from classes.GolayDecoder import GolayDecoder

In [3]:
gf            = GaloisField(1, 0b11)
encoder_model = GolayEncoder()
k             = encoder_model._k
n             = encoder_model._n

Field Closed Succesfully!, 1 Non-Zero Elements


## Verifify code properties

In [9]:
G2412G  = np.hstack((np.eye(k), encoder_model._G2412B)).astype(np.uint8)
G2412H  = np.hstack((encoder_model._G2412B.T, np.eye(n-k))).astype(np.uint8)

# test code properties
g_ht    = gf.mat_mul(G2412G, G2412H.T)
b_2     = gf.mat_mul(encoder_model._G2412B, encoder_model._G2412B)
print(f"Property: G*HT = 0")
print(f"{32*'='}")
print(g_ht)
print()
print(f"Property: B*B = I")
print(f"{32*'='}")
print(b_2)

Property: G*HT = 0
[[0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]]

Property: B*B = I
[[1 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 1]]


In [4]:
encoder_output = None

words = []

for w in range(2**k):
    w  = gf.do_unpack(w, bit_width= k)
    words.append(w)
    cw = encoder_model.encode(w)
    encoder_output = cw if encoder_output is None else np.vstack((encoder_output, cw))


n_codewords = len(encoder_output)  

In [5]:
words
encoder_output

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 1, 0],
       ...,
       [1, 1, 1, ..., 1, 0, 1],
       [1, 1, 1, ..., 1, 1, 0],
       [1, 1, 1, ..., 1, 1, 1]], shape=(4096, 24), dtype=uint8)

## Get weight distribution

In [9]:
# get weight distribution, match with golay 24,12 distribution
weights = np.full(n+1, -1).astype(np.int32)
for cw in encoder_output:
    w = gf.hamming_weight(cw)
    weights[w] = 1 if weights[w] == -1 else weights[w]+1

weights

print(f"Golay 24,12 Weight Distribution:")
print(f"{32*'='}")
for i, w in enumerate(weights):
    if w != -1:
        print(f"W{i}\t: {w}")

Golay 24,12 Weight Distribution:
W0	: 1
W8	: 759
W12	: 2576
W16	: 759
W24	: 1


## Write test vectors to .svh

In [17]:
def write_sv_array(buf, name, width_label, width_bits, values, comment=None):
    """Write a plain SV array decl: bit [width_label] name [len(values)] = '{ ... };
    `width_label` is the bracket text (e.g. "NB_CODEWORD-1:0"), or None for a 1-bit `bit` array."""
    if comment:
        buf.write(f"\t// {comment}\n")
    decl = f"bit {name}" if width_label is None else f"bit [{width_label}] {name}"
    buf.write(f"\t{decl} [{len(values)}] = '{{\n")
    for i, v in enumerate(values):
        literal = f"1'b{int(v)}" if width_label is None else f"{width_bits}'b{v:0{width_bits}b}"
        sep = "};\n" if i == len(values) - 1 else ",\n"
        buf.write(f"\t\t{literal}\t{sep}")
    buf.write("\n")

In [18]:
from io import StringIO

filenames = [
    "outputs/encoding_vectors_test/encoded_words_testing_golay_encoder.svh",
    "../implem/golay_encoder/encoder/golay_encoder.uvm/sequences/encoder_testing/encoded_words_testing_golay_encoder.svh"
]

file_content = StringIO()

# Input messages
write_sv_array(file_content, "words", "NB_WORD-1:0", 12,
               [gf.do_pack(w) for w in words],
               comment="Input messages")

# Encoded codewords
write_sv_array(file_content, "encoder_output", "NB_CODEWORD-1:0", 24,
               [gf.do_pack(cw) for cw in encoder_output],
               comment="Encoded codewords")

content = file_content.getvalue()

for filename in filenames:
    with open(filename, "w") as file:
        file.write(content)